In [0]:
import random
from datetime import datetime
from cdp_dq_framework.utils import strip_glob

In [0]:
# Retrieve job parameters
bucket = dbutils.widgets.get("bucket_name")
file_key = dbutils.widgets.get("bucket_key")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

In [0]:
now = datetime.now()
file_arrival_time = now.strftime('%Y-%m-%d %H:%M:%S')
excn_id = f"{now.year}{now.month}{now.day}{now.hour}{now.minute}{now.second}{random.randint(101,999)}"
file_name = file_key.split("/")[-1]
matched = None

In [0]:
print(f"{bucket=}")
print(f"{file_key=}")
print(f"{file_name=}")

In [0]:
query = f"SELECT * FROM {catalog}.{schema}.dc_entity_mstr"
print(f"{query=}")
for row in spark.sql(query).collect():
	prefix = strip_glob(row.file_nm)
	if prefix and prefix in file_name:
		if matched is None or len(strip_glob(matched.file_nm)) < len(prefix):
			matched = row
if matched is None:
	dbutils.jobs.taskValues.set("data_found", "NO")
	raise Exception(f"No dc_entity_mstr entry matches '{file_name}'")  # legacy hard-stop

In [0]:
# publish for downstream tasks + record arrival
tv = dbutils.jobs.taskValues
task_values = {
    "data_found": "YES",
    "bucket": bucket,
    "file_key": file_key,
    "app_id": matched.app_id.upper(),
    "file_name": file_name,
    "entity_id": matched.entity_id.upper(),
    "excn_id": excn_id,
    "entity_name": matched.entity_nm.upper(),
    "service_type": matched.service,
    "pre_contract_path": matched.raw_pre_processing_path,
    "pre_contract_file_path": f"s3://{matched.raw_pre_processing_path}/{file_name}",
    "post_process_path": matched.raw_post_processing_path,
    "file_arrival_time": file_arrival_time,
    "topic_arn": matched.email
}
for key, val in task_values.items():
    tv.set(key=key, value=val)